# ENGRAM: choose the trigger threshold on VALIDATION people

The paper's trigger threshold (0.6) was picked by looking at the test clips. Reviewers rightly object.
This notebook picks it the fair way: on **people P04, P21, P22, P27, P31**, the validation people of the
small model (never used for training, never used for the test table).

**The rule is fixed before the run:** for each threshold, count actions found by the trigger minus actions found by
"every n-th window" with the same number of big-model calls. The largest gain wins (ties → fewer calls).
The test-people results are then re-scored on the laptop at the chosen threshold, without the GPU.

**Settings (right panel):** Accelerator **GPU T4** · Internet **On** · Input: dataset **mnemofyp** (latest version).

**How to run:** top right **Save Version** → **Save & Run All (Commit)** → Save. About 1 hour; you can close the browser.
When the version is finished, open it → **Output** → download `tau_val_results.zip`.

## 1. Code + packages (~3 min)

In [ ]:
import os, shutil, glob, sys, subprocess
WORK = '/kaggle/working'
CODE = f'{WORK}/mnemo'
code_dir = code_zip = None
for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
    if 'mnemo.py' in files and 'modules' in dirs:
        code_dir = root
    if 'mnemo_code.zip' in files:
        code_zip = os.path.join(root, 'mnemo_code.zip')
print('code folder:', code_dir, '| zip:', code_zip)
assert code_dir or code_zip, 'Code not found: add the mnemofyp dataset (Add Input) and wait until it is attached'
shutil.rmtree(CODE, ignore_errors=True)
if code_dir:
    shutil.copytree(code_dir, CODE)
else:
    os.makedirs(CODE)
    !cd "{CODE}" && unzip -q "{code_zip}"
%cd {CODE}
assert os.path.exists('evaluation/choose_threshold.py'), 'OLD code: upload the new mnemo_code.zip as a new version of mnemofyp'
!pip install -q -U "transformers>=4.57" accelerate mediapipe ultralytics
import torch, numpy, transformers
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE: switch Accelerator to GPU T4!')
print('transformers', transformers.__version__, '| numpy', numpy.__version__)

## 2. Start downloading the validation videos (runs in the background)
30 random minutes from the 5 validation people (not the busiest minutes: ordinary everyday stretches, like the test).

In [ ]:
import urllib.request
EK = f'{WORK}/epic'
CLIPS_DIR = f'{WORK}/clips_val'
os.makedirs(EK, exist_ok=True); os.makedirs(CLIPS_DIR, exist_ok=True)
A = 'https://raw.githubusercontent.com/epic-kitchens/epic-kitchens-100-annotations/master'
D = 'https://raw.githubusercontent.com/epic-kitchens/epic-kitchens-download-scripts/master/data'
for url in [f'{A}/EPIC_100_train.csv', f'{A}/EPIC_100_verb_classes.csv', f'{A}/EPIC_100_noun_classes.csv',
            f'{A}/EPIC_100_video_info.csv', f'{D}/epic_55_splits.csv']:
    dst = f'{EK}/{url.rsplit("/", 1)[1]}'
    if not os.path.exists(dst):
        urllib.request.urlretrieve(url, dst)
ANN = f'{EK}/EPIC_100_train.csv'
N_CLIPS = 30
VAL_P = ['P04', 'P21', 'P22', 'P27', 'P31']     # validation people of the small model (stored in hoi_tcn.pt)
SEED = 11
FETCH_LOG = f'{WORK}/fetch.log'
FETCH = subprocess.Popen([sys.executable, '-m', 'tools.fetch_epic_windows', '--data', EK, '--out', CLIPS_DIR,
                          '--select', 'random', '--windows', str(N_CLIPS), '--window-s', '60', '--workers', '8',
                          '--participants', *VAL_P, '--seed', str(SEED)],
                         cwd=CODE, stdout=open(FETCH_LOG, 'w'), stderr=subprocess.STDOUT)
print('download started in the background → continue with step 3')

## 3. Load the models (~5 min)

In [ ]:
import time
import config as C
from modules.vlm_logger import VLMDescriber
from modules.hoi_gate import HOIGate
t = time.time()
describer = VLMDescriber(C.VLM_MODEL)
gate = HOIGate('models/hoi_tcn.pt')
print(f'loaded in {time.time() - t:.0f}s | prompt version {C.VLM_PROMPT_VERSION}')

## 4. Wait for the downloads to finish

In [ ]:
FETCH.wait()
print(open(FETCH_LOG).read()[-1200:])
CLIPS = sorted(glob.glob(f'{CLIPS_DIR}/*_w*.mp4'))
print(len(CLIPS), 'clips:', [os.path.basename(c) for c in CLIPS])
assert all(os.path.basename(c)[:3] in VAL_P for c in CLIPS), 'a clip from a non-validation person slipped in'

## 5. Run the big model on every window, then score each threshold (~1 h)
Same code as the test benchmark, without the questions. Finished clips are cached: if Kaggle stops, run this cell again.

In [ ]:
from evaluation.benchmark import run_benchmark
from evaluation.choose_threshold import report
from IPython.display import Markdown, display
RESULTS = f'{WORK}/tau_val_{C.VLM_PROMPT_VERSION}_seed{SEED}'
results, summary = run_benchmark(CLIPS, describer, gate, ANN, EK, RESULTS,
                                 thresholds=(0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 0.8), qa=False)
choice = report(results)
with open(f'{RESULTS}/threshold_choice.md', 'w') as f:
    f.write(choice + '\n')
display(Markdown(choice))

## 6. Save
Download `tau_val_results.zip` (tables + cached big-model text; no video).

In [ ]:
shutil.make_archive(f'{WORK}/tau_val_results', 'zip', RESULTS)
print(open(f'{RESULTS}/threshold_choice.md').read())
os.chdir(WORK)
from IPython.display import FileLink
display(FileLink('tau_val_results.zip'))